# 08. Tabla maestra consolidada y comparabilidad del SVM

Este notebook consolida la tabla maestra de resultados (`results/master_table_112.csv`) y verifica la comparabilidad de las filas de SVM, corridas en Colab con `cuml.svm.SVC`, frente a las demás, que usan `sklearn.svm.SVC`. Los números de este notebook son los de la corrida 5 x 5 (5 folds externos x 5 internos). KNN también corrió en la GPU de Colab (cuML), y se documenta aparte su diferencia con scikit-learn.

## 1. Tabla maestra de las 112 combinaciones

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json

import numpy as np
import pandas as pd

from src import config
from src import master_table as mt
from src import stats_comparison as sc

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

master, best = mt.save_master_tables()
print(f"Tabla maestra: {len(master)} filas -> {mt.MASTER_112_PATH.relative_to(config.PROJECT_ROOT)}")
print(f"Selección por modelo: {len(best)} filas -> {mt.BEST_PER_MODEL_PATH.relative_to(config.PROJECT_ROOT)}")
print(f"Diferencia crítica de Nemenyi (k = {len(master)}, {config.N_OUTER_FOLDS} folds): {master.attrs['critical_difference']:.1f}")
master.groupby(["model", "source"]).size().unstack(fill_value=0)

Tabla maestra: 112 filas -> results\master_table_112.csv
Selección por modelo: 7 filas -> results\best_per_model.csv
Diferencia crítica de Nemenyi (k = 112, 5 folds): 89.4


source,colab_gpu_cuml,local_cpu_sklearn
model,,
decision_tree,0,16
knn,16,0
logistic_regression,0,16
naive_bayes,0,16
random_forest,0,16
svm,16,0
xgboost,0,16


La tabla `results/master_table_112.csv` une las 96 filas de `results/experiments_master_5x5.csv` (80 de la CPU local y 16 de KNN en Colab), las 16 de SVM (`results/svm_gpu_results_colab_5x5.csv`) y las métricas complementarias (`experiments_posthoc_metrics.csv`). Cada fila trae el origen del cálculo (`source`), la semilla, el tiempo y las métricas por fold con su media y desviación estándar entre los 5 folds externos, **con `ddof=1` en todas** (la tabla de la corrida 3 x 3 usaba `ddof=0`). También trae el rango medio de F1 entre las 112 combinaciones y si la combinación queda a menos de la diferencia crítica de Nemenyi (89.4) de la mejor. Los tiempos de SVM y de KNN son de GPU y se marcan con `time_comparable = False`. El PR-AUC de SVM viene del reajuste en CPU (ver la columna `pr_auc_source`), y el de KNN del reajuste con scikit-learn de `posthoc_metrics`. A continuación, las 15 primeras filas ordenadas por rango medio de F1.

In [2]:
columns = ["model", "technique", "method", "source", "random_state", "total_time_seconds", "time_comparable",
           "f1_mean", "f1_std", "precision_mean", "recall_mean", "roc_auc_mean", "roc_auc_std", "pr_auc_mean",
           "f1_avg_rank", "within_nemenyi_cd_of_best"]
master[columns].head(15).round(4)

,model,technique,method,source,random_state,total_time_seconds,time_comparable,f1_mean,f1_std,precision_mean,recall_mean,roc_auc_mean,roc_auc_std,pr_auc_mean,f1_avg_rank,within_nemenyi_cd_of_best
0,xgboost,class_weight,bayesian_optuna,local_cpu_sklearn,42,309.6490,True,0.5466,0.0091,0.4927,0.6149,0.7828,0.0089,0.5567,2.4,True
1,random_forest,class_weight,random_search,local_cpu_sklearn,42,657.6279,True,0.5452,0.0077,0.4974,0.6041,0.7831,0.0095,0.5606,3.8,True
2,random_forest,class_weight,grid_search,local_cpu_sklearn,42,717.8981,True,0.5448,0.0095,0.4962,0.6039,0.7846,0.0089,0.5621,4.2,True
3,random_forest,class_weight,bayesian_optuna,local_cpu_sklearn,42,1689.4361,True,0.5441,0.0080,0.4944,0.6053,0.7825,0.0092,0.5602,5.4,True
4,xgboost,class_weight,grid_search,local_cpu_sklearn,42,774.9741,True,0.5429,0.0086,0.4783,0.6278,0.7823,0.0093,0.5483,6.2,True
5,random_forest,class_weight,genetic_deap,local_cpu_sklearn,42,8300.8405,True,0.5434,0.0105,0.4971,0.5995,0.7822,0.0087,0.5606,6.4,True
6,xgboost,class_weight,genetic_deap,local_cpu_sklearn,42,1181.0180,True,0.5422,0.0111,0.4853,0.6148,0.7830,0.0091,0.5616,8.9,True
7,xgboost,class_weight,random_search,local_cpu_sklearn,42,246.1009,True,0.5414,0.0107,0.4959,0.5968,0.7801,0.0095,0.5533,9.6,True
8,svm,smote,random_search,colab_gpu_cuml,42,1434.8477,False,0.5387,0.0048,0.4795,0.6148,0.7638,0.0096,0.5132,12.0,True
9,svm,class_weight,bayesian_optuna,colab_gpu_cuml,42,1557.0969,False,0.5385,0.0077,0.4765,0.6192,0.7649,0.0096,0.5044,13.2,True


In [3]:
# Control de integridad: una fila por combinación, sin vacíos en las columnas clave.
assert len(master) == 112 and not master.duplicated(["model", "technique", "method"]).any()
assert master[["f1_mean", "f1_std", "roc_auc_mean", "total_time_seconds"]].notna().all().all()
print("OK: 112 combinaciones únicas; F1, ROC-AUC y tiempo completos.")
print("Filas sin PR-AUC:", int(master["pr_auc_mean"].isna().sum()))

OK: 112 combinaciones únicas; F1, ROC-AUC y tiempo completos.
Filas sin PR-AUC: 0


## 2. Mejor combinación por modelo y motivo de la selección

Para cada modelo se elige la combinación (técnica de balanceo y método de optimización) con mayor F1 medio en el bucle externo. La tabla se guarda en `results/best_per_model.csv`, con el motivo de cada selección. La columna `within_nemenyi_cd_of_global_best` indica si la combinación está dentro del conjunto de finalistas de la comparación de Friedman y Nemenyi (a menos de la diferencia crítica de la mejor global).

In [4]:
best[["model", "technique", "method", "f1_mean_std", "roc_auc_mean_std", "gap_to_runner_up",
      "runner_up", "within_nemenyi_cd_of_global_best", "is_global_best", "source"]].round(4)

,model,technique,method,f1_mean_std,roc_auc_mean_std,gap_to_runner_up,runner_up,within_nemenyi_cd_of_global_best,is_global_best,source
0,xgboost,class_weight,bayesian_optuna,0.5466 ± 0.0091,0.7828 ± 0.0089,0.0037,class_weight|grid_search,True,True,local_cpu_sklearn
1,random_forest,class_weight,random_search,0.5452 ± 0.0077,0.7831 ± 0.0095,0.0005,class_weight|grid_search,True,False,local_cpu_sklearn
2,svm,smote,random_search,0.5387 ± 0.0048,0.7638 ± 0.0096,0.0001,class_weight|genetic_deap,True,False,colab_gpu_cuml
3,knn,class_weight,random_search,0.5264 ± 0.0122,0.7629 ± 0.0081,0.0012,class_weight|grid_search,True,False,colab_gpu_cuml
4,decision_tree,class_weight,bayesian_optuna,0.5254 ± 0.0043,0.7569 ± 0.0101,0.0003,class_weight|genetic_deap,True,False,local_cpu_sklearn
5,logistic_regression,class_weight,genetic_deap,0.5182 ± 0.0104,0.7449 ± 0.0147,0.0017,class_weight|bayesian_optuna,True,False,local_cpu_sklearn
6,naive_bayes,smote,grid_search,0.4987 ± 0.0078,0.7454 ± 0.0115,0.0001,smote|bayesian_optuna,True,False,local_cpu_sklearn


In [5]:
for _, r in best.iterrows():
    print(f"- {r['model']}: {r['selection_reason']}")

- xgboost: Mayor F1 medio externo entre las 16 combinaciones de xgboost; ventaja de 0.0037 sobre class_weight|grid_search (0.4 desviaciones estándar entre folds).
- random_forest: Mayor F1 medio externo entre las 16 combinaciones de random_forest; ventaja de 0.0005 sobre class_weight|grid_search (0.1 desviaciones estándar entre folds).
- svm: Mayor F1 medio externo entre las 16 combinaciones de svm; ventaja de 0.0001 sobre class_weight|genetic_deap (0.0 desviaciones estándar entre folds).
- knn: Mayor F1 medio externo entre las 16 combinaciones de knn; ventaja de 0.0012 sobre class_weight|grid_search (0.1 desviaciones estándar entre folds).
- decision_tree: Mayor F1 medio externo entre las 16 combinaciones de decision_tree; ventaja de 0.0003 sobre class_weight|genetic_deap (0.1 desviaciones estándar entre folds).
- logistic_regression: Mayor F1 medio externo entre las 16 combinaciones de logistic_regression; ventaja de 0.0017 sobre class_weight|bayesian_optuna (0.2 desviaciones estánda

**Lectura.**
- La mejor combinación global es XGBoost con `class_weight` y `bayesian_optuna` (F1 0.5466 ± 0.0091), seguida de Random Forest con `class_weight` y `random_search` (0.5452 ± 0.0077).
- Las siete combinaciones elegidas están dentro de la diferencia crítica de la mejor global. Esto **no prueba equivalencia**: la diferencia crítica es 89.4 y la máxima diferencia de rangos, 108.3, así que Nemenyi separa solo combinaciones muy alejadas en el ranking y no puede ordenar las cercanas a la mejor.
- La ventaja de cada combinación elegida sobre la segunda mejor de su mismo modelo es de entre 0.0001 y 0.0037 de F1, menor que la desviación entre folds. La elección dentro de cada modelo es prácticamente un desempate. En cinco de los siete modelos la técnica elegida es `class_weight` y en los otros dos es SMOTE.

## 3. ¿Son comparables las 16 filas de SVM?

Repetir las búsquedas de SVM en CPU no es viable (en la corrida inicial 3 x 3, `svm/none/genetic_deap` tardó más de 4 horas en la CPU local, y con 5 x 5 sería varias veces más). Lo que sí se puede verificar es la parte de la comparación que depende de la evaluación y no de la búsqueda:

1. Se toman los hiperparámetros que eligió el bucle interno de Colab en cada fold externo.
2. Se reajusta, con el mismo pipeline del resto del proyecto (`balancing.build_pipeline`, con `sklearn.svm.SVC`), sobre los mismos folds externos, y se mide el F1 de prueba.
3. Se compara con el F1 por fold que reportó Colab.

Si los dos coinciden, las filas de SVM miden lo mismo que las demás. Lo que **no** se puede verificar con esto es que la búsqueda de cuML habría elegido los mismos hiperparámetros que la de libsvm, ni los tiempos.

In [6]:
check = mt.load_svm_check()
if check is None or len(check) < 16:
    raise RuntimeError("Falta el reajuste local de SVM: correr `python -m src.svm_comparability`.")
check = check.reset_index()
cmp = check[["technique", "method", "f1_colab_mean", "f1_local_mean", "f1_max_abs_diff_per_fold",
             "roc_auc_max_abs_diff_per_fold"]].copy()
cmp["dif_media_f1 (local - colab)"] = cmp["f1_local_mean"] - cmp["f1_colab_mean"]
cmp = cmp.sort_values(["technique", "method"]).reset_index(drop=True)
display(cmp.round(4))

diff = cmp["dif_media_f1 (local - colab)"]
by_tech = cmp.groupby("technique")["dif_media_f1 (local - colab)"].mean().round(4)
print(f"Diferencia media de F1 (local - Colab), 16 filas: {diff.mean():+.4f} (mín {diff.min():+.4f}, máx {diff.max():+.4f})")
print("Por técnica:", by_tech.to_dict())
print(f"Máxima diferencia absoluta por fold: {cmp['f1_max_abs_diff_per_fold'].max():.4f}")
print(f"Filas con diferencia por fold mayor que 0.01: {(cmp['f1_max_abs_diff_per_fold'] > 0.01).sum()} de 16")
print(f"Mediana de la desviación estándar entre folds del F1 (resto de modelos): "
      f"{master.loc[master['model'] != 'svm', 'f1_std'].median():.4f}")

,technique,method,f1_colab_mean,f1_local_mean,f1_max_abs_diff_per_fold,roc_auc_max_abs_diff_per_fold,dif_media_f1 (local - colab)
0,adasyn,bayesian_optuna,0.5298,0.5390,0.0117,0.0008,0.0092
1,adasyn,genetic_deap,0.5301,0.5379,0.0134,0.0005,0.0079
2,adasyn,grid_search,0.5287,0.5342,0.0135,0.0010,0.0055
3,adasyn,random_search,0.5293,0.5384,0.0133,0.0008,0.0091
4,class_weight,bayesian_optuna,0.5385,0.5440,0.0088,0.0000,0.0056
5,class_weight,genetic_deap,0.5386,0.5446,0.0098,0.0009,0.0059
6,class_weight,grid_search,0.5372,0.5446,0.0115,0.0006,0.0074
7,class_weight,random_search,0.5385,0.5440,0.0086,0.0000,0.0055
8,none,bayesian_optuna,0.4446,0.4668,0.0264,0.0002,0.0221
9,none,genetic_deap,0.4390,0.4541,0.0237,0.0005,0.0152


Diferencia media de F1 (local - Colab), 16 filas: +0.0089 (mín +0.0033, máx +0.0221)
Por técnica: {'adasyn': 0.0079, 'class_weight': 0.0061, 'none': 0.0173, 'smote': 0.0044}
Máxima diferencia absoluta por fold: 0.0264
Filas con diferencia por fold mayor que 0.01: 9 de 16
Mediana de la desviación estándar entre folds del F1 (resto de modelos): 0.0104


**Lectura.**
- **Las filas de SVM de Colab no reproducen exactamente el F1 de scikit-learn.** Con los mismos hiperparámetros, los mismos folds y el mismo pipeline, el F1 reajustado en CPU es mayor en las 16 filas, en promedio en 0.009 (entre 0.003 y 0.022). La diferencia es mayor sin balanceo (0.017) y menor con SMOTE (0.004) y `class_weight` (0.006). En 9 de las 16 filas hay al menos un fold con diferencia mayor que 0.01. La mediana de la desviación entre folds de los demás modelos es 0.010, de modo que la diferencia es algo menor que una desviación estándar.
- El ROC-AUC casi no cambia (diferencia máxima por fold de 0.001), así que el orden en que el modelo clasifica a los clientes es el mismo y la discrepancia viene del punto de corte de las predicciones. Es consistente con que el solver y la calibración de probabilidades de cuML no son idénticos a los de libsvm, pero es una hipótesis que no se verificó.
- **Conclusión:** las filas de SVM de Colab no son estrictamente comparables en F1 con las demás, y el sesgo siempre va en contra de SVM. Los tiempos tampoco son comparables. Por eso el análisis siguiente se hace con tres versiones de la comparación.
- **KNN también corrió en la GPU de Colab (cuML).** No se rehízo la búsqueda en CPU, pero `posthoc_metrics` reajustó con scikit-learn los hiperparámetros elegidos: las clases predichas coinciden en 99.98 % a 100 % de los casos y el F1 por fold difiere hasta 0.0066 respecto al de Colab (en las 96 filas no SVM, ese es el máximo). Las probabilidades difieren por precisión float32 y empates entre vecinos. Es una diferencia menor que la de SVM y no sistemática.

## 4. Sensibilidad de la comparación estadística al tratamiento de SVM

Se repite la prueba de Friedman sobre el F1 por fold en tres escenarios: **A**, las 112 filas tal como se reportaron; **B**, solo las 96 combinaciones locales, sin SVM; **C**, las 112 filas con el F1 por fold de SVM reemplazado por el del reajuste en CPU. Si las conclusiones no cambian entre escenarios, no dependen del tratamiento de SVM.

In [7]:
def friedman_summary(table, label):
    matrix = sc.build_performance_matrix_from_table(table, metric="f1")
    fr = sc.friedman_test(matrix)
    cd = sc.compute_critical_difference(k=fr["k_combinations"], n_blocks=fr["n_blocks"])
    ranks = fr["avg_ranks"]
    top = ranks.index[:3].tolist()
    spread = float(ranks.max() - ranks.min())
    return {
        "escenario": label, "combinaciones": fr["k_combinations"],
        "chi2": fr["statistic"], "p": fr["p_value"], "CD Nemenyi": cd,
        "máx. dif. de rangos": spread, "Nemenyi con potencia": bool(spread > cd),
        "mejor": top[0], "2.º": top[1], "3.º": top[2],
    }, ranks


# A: las 112 filas tal como se reportaron (SVM de Colab).
A = master
# B: solo las 96 combinaciones locales (sin SVM).
B = master[master["model"] != "svm"]
# C: las 112 filas, con el F1 por fold de SVM reemplazado por el del reajuste en CPU.
C = master.copy()
is_svm = C["model"] == "svm"
C.loc[is_svm, "f1_per_fold"] = C.loc[is_svm, "f1_cpu_refit_per_fold"]
C.loc[is_svm, "f1_mean"] = C.loc[is_svm, "f1_cpu_refit_mean"]

rows, ranks_by_scenario = [], {}
for label, table in [("A. 112 filas, SVM de Colab", A), ("B. 96 filas, sin SVM", B), ("C. 112 filas, SVM reajustado en CPU", C)]:
    summary, ranks = friedman_summary(table, label)
    rows.append(summary)
    ranks_by_scenario[label] = ranks
sens = pd.DataFrame(rows).set_index("escenario")
sens

,combinaciones,chi2,p,CD Nemenyi,máx. dif. de rangos,Nemenyi con potencia,mejor,2.º,3.º
escenario,,,,,,,,,
"A. 112 filas, SVM de Colab",112,534.600162,1.123308e-56,89.372108,108.3,True,xgboost|class_weight|bayesian_optuna,random_forest|class_weight|random_search,random_forest|class_weight|grid_search
"B. 96 filas, sin SVM",96,453.809081,3.363868e-48,75.490309,92.3,True,xgboost|class_weight|bayesian_optuna,random_forest|class_weight|random_search,random_forest|class_weight|grid_search
"C. 112 filas, SVM reajustado en CPU",112,535.948560,6.562144e-57,89.372108,106.3,True,xgboost|class_weight|bayesian_optuna,random_forest|class_weight|random_search,svm|class_weight|grid_search


In [8]:
# Ranking de las mejores combinaciones de cada modelo en cada escenario (mejor F1 medio por modelo).
def best_by_model(table):
    idx = table.groupby("model")["f1_mean"].idxmax()
    return table.loc[idx].sort_values("f1_mean", ascending=False)[["model", "technique", "method", "f1_mean"]]

for label, table in [("A", A), ("B", B), ("C", C)]:
    print(f"\nEscenario {label}:")
    display(best_by_model(table).reset_index(drop=True).round(4))


Escenario A:


,model,technique,method,f1_mean
0,xgboost,class_weight,bayesian_optuna,0.5466
1,random_forest,class_weight,random_search,0.5452
2,svm,smote,random_search,0.5387
3,knn,class_weight,random_search,0.5264
4,decision_tree,class_weight,bayesian_optuna,0.5254
5,logistic_regression,class_weight,genetic_deap,0.5182
6,naive_bayes,smote,grid_search,0.4987



Escenario B:


,model,technique,method,f1_mean
0,xgboost,class_weight,bayesian_optuna,0.5466
1,random_forest,class_weight,random_search,0.5452
2,knn,class_weight,random_search,0.5264
3,decision_tree,class_weight,bayesian_optuna,0.5254
4,logistic_regression,class_weight,genetic_deap,0.5182
5,naive_bayes,smote,grid_search,0.4987



Escenario C:


,model,technique,method,f1_mean
0,xgboost,class_weight,bayesian_optuna,0.5466
1,random_forest,class_weight,random_search,0.5452
2,svm,class_weight,grid_search,0.5446
3,knn,class_weight,random_search,0.5264
4,decision_tree,class_weight,bayesian_optuna,0.5254
5,logistic_regression,class_weight,genetic_deap,0.5182
6,naive_bayes,smote,grid_search,0.4987


**Lectura.**
- **La conclusión estadística no cambia.** En los tres escenarios Friedman es significativo ($p<10^{-47}$) y Nemenyi tiene algo de potencia: la máxima diferencia de rangos medios (108.3, 92.3 y 106.3) supera la diferencia crítica (89.4, 75.5 y 89.4), de modo que separa los extremos del ranking pero no las combinaciones de cabeza. Las dos primeras combinaciones son las mismas en los tres escenarios: XGBoost con `class_weight` y `bayesian_optuna`, y Random Forest con `class_weight` y `random_search`.
- **El ranking de modelos cambia solo en la posición de SVM.** Con el SVM de Colab (escenario A), la mejor combinación de SVM tiene F1 0.5387 (SMOTE, `random_search`) y queda tercera entre los modelos. Con el reajuste en CPU (escenario C) sube a 0.5446 (`class_weight` con `grid_search`), a 0.0020 de XGBoost (0.5466), y la tercera combinación del ranking global pasa de Random Forest con `grid_search` a SVM con `class_weight` y `grid_search`. Este cambio es desfavorable para la versión de Colab, que subestimaba a SVM, y se reporta así. XGBoost sigue siendo el mejor, y la diferencia con SVM está muy por debajo de la desviación entre folds.
- Sin SVM (escenario B) el orden de los otros seis modelos es el mismo que con SVM.

## Limitaciones

- El reajuste en CPU usa los hiperparámetros que eligió Colab. No demuestra que una búsqueda en CPU habría elegido lo mismo, así que el F1 reajustado es el de la mejor configuración encontrada por cuML evaluada con libsvm, no el de una búsqueda completa en CPU.
- El F1 reajustado se calculó con `probability=False` (las predicciones de `SVC` usan el signo de la función de decisión en ambos casos). El ROC-AUC y el PR-AUC de SVM en este reajuste salen de `decision_function` y no de `predict_proba`.
- En la tabla maestra se conservan los valores de Colab como fueron reportados (columnas `f1_*`) y el reajuste va aparte (`f1_cpu_refit_*`), para no reescribir resultados originales.
- Los tiempos de SVM y de KNN (GPU T4, otro hardware) no son comparables con los del resto (CPU local).
- Con 5 folds, la prueba de Nemenyi solo separa combinaciones muy alejadas en el ranking, y los intervalos de las métricas son anchos. La comparación entre las combinaciones de cabeza es descriptiva.